Modeling Features: Linear Light Profiles
========================================

A "linear light profile" is a variant of a standard light profile where the `intensity` parameter is solved for
via linear algebra every time the model is fitted to the data. This uses a process called an "inversion" and it
always computes the `intensity` values that give the best fit to the data (e.g. maximize the likelihood)
given the light profile's other parameters.

Based on the advantages below, we recommended you always use linear light profiles to fit models over standard
light profiles!

__Contents__

- **Advantages & Disadvantages:** Benefits and drawbacks of linear light profiles.
- **Positive Only Solver:** How a positive solution to the light profile intensities is ensured.
- **Dataset & Mask:** Standard set up of imaging dataset that is fitted.
- **Fit:** Perform a fit to a dataset using linear light profile with inputs for other light profile parameters.
- **Intensities:** Access the solved for intensities of light profiles from the fit.
- **Model:** Composing a model using linear light profiles and how it changes the number of free parameters.
- **Search & Analysis:** Standard set up of non-linear search and analysis.
- **Run Time:** Profiling of linear light profile run times and discussion of how they compare to standard light profiles.
- **Model-Fit:** Performs the model fit using standard API.
- **Result & Intensities:** Linear light profiles results, including how to access light profiles with solved for intensity values.
- **Visualization:** Plotting images of model-fits using linear light profiles.
- **Linear Objects (Source Code)**: Internal source code implementation of linear light profiles (for contributors).

__Advantages__

Each light profile's `intensity` parameter is therefore not a free parameter in the model-fit, reducing the
dimensionality of non-linear parameter space by the number of light profiles (in this example by 2 dimensions).

This also removes the degeneracies that occur between the `intensity` and other light profile parameters
(e.g. `effective_radius`, `sersic_index`), which are difficult degeneracies for the non-linear search to map out
accurately. This produces more reliable lens model results and the fit converges in fewer iterations, speeding up the
overall analysis.

The inversion has a relatively small computational cost, thus we reduce the model complexity without much slow-down and
can therefore fit models more reliably and faster!

__Disadvantages__

Although the computation time of the inversion is small, it is not non-negligible. It is approximately 3-4x slower
than using a standard light profile.

The gains in run times due to the simpler non-linear parameter space therefore are somewhat balanced by the slower
likelihood calculation.

__Positive Only Solver__

Many codes which use linear algebra typically rely on a linear algebra solver which allows for positive and negative
values of the solution (e.g. `np.linalg.solve`), because they are computationally fast.

This is problematic, as it means that negative surface brightnesses values can be computed to represent a galaxy's
light, which is clearly unphysical.

**PyAutoGalaxy** uses a positive only linear algebra solver which has been extensively optimized to ensure it is as fast
as positive-negative solvers. This ensures that all light profile intensities are positive and therefore physical.

__Model__

This script fits an `Imaging` dataset of a galaxy with a model where:

 - The galaxy's light is a linear parametric `Sersic` bulge and `Exponential` disk.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/modeling.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autogalaxy as ag
import autogalaxy.plot as aplt

__Dataset__

Load and plot the galaxy dataset `simple` via .fits files.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if ag.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/simulator.py"],
        check=True,
    )


dataset = ag.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of the galaxy.

In [ ]:
mask = ag.Mask2D.circular(
    shape_native=dataset.shape_native, pixel_scales=dataset.pixel_scales, radius=3.0
)

dataset = dataset.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=dataset)

__Over Sampling__

Apply adaptive over sampling to ensure the calculation is accurate, you can read up on over-sampling in more detail via
the `autogalaxy_workspace/*/guides/advanced/over_sampling.ipynb` notebook.

In [ ]:
over_sample_size = ag.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=dataset.grid,
    sub_size_list=[8, 4, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(0.0, 0.0)],
)

dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

aplt.subplot_imaging_dataset(dataset=dataset)

__Model__

We compose our model where in this example:

 - The galaxy's bulge is a linear parametric `Sersic` bulge [6 parameters]. 
 - The galaxy's disk is a linear parametric `Exponential` disk, whose centre is aligned with the bulge [3 parameters].
 
The number of free parameters and therefore the dimensionality of non-linear parameter space is N=9.

Note how both light profiles use linear light profiles, meaning that the `intensity` parameter of both is no longer a 
free parameter in the fit. This means the overall number of free parameters is reduced by two compared to if 
standard light profiles were used.

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)
disk = af.Model(ag.lp_linear.Exponential)
bulge.centre = disk.centre

galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge, disk=disk)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

This confirms that the light profiles of the lens and source galaxies do not include an `intensity` parameter.

In [ ]:
print(model.info)

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start_here.py` for a
full description).

In the `start_here.py` example 100 live points (`n_live=100`) were used to sample parameter space. For the linear
light profiles this is reduced to 75, as the simpler parameter space means we need fewer live points to map it out
accurately. This will lead to faster run times.

In [ ]:
search = af.Nautilus(
    path_prefix=Path("imaging") / "features",
    name="linear_light_profiles",
    unique_tag=dataset_name,
    n_live=75,
    n_batch=50,  # GPU galaxy model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Analysis__

Create the `AnalysisImaging` object defining how the via Nautilus the model is fitted to the data.

In [ ]:
analysis = ag.AnalysisImaging(
    dataset=dataset,
    use_jax=True,
)

__VRAM__

The `modeling` example explains how VRAM is used during GPU-based fitting and how to
print the estimated VRAM required by a model.

For each linear light profile in the model extra is used VRAM. For 3-10 linear Sersic light profiles this is a tiny 
amount of VRAM (e.g. < 10MB  per batched likelihood). Even for large batch sizes (e.g. over 100) you probably 
will not use enough VRAM to require monitoring.

__Run Time__

For standard light profiles, the log likelihood evaluation time is of order ~0.01 seconds for this dataset.

For linear light profiles, the log likelihood evaluation increases to around ~0.05 seconds per likelihood evaluation.
This is still fast, but it does mean that the fit may take around five times longer to run.

However, because two free parameters have been removed from the model (the `intensity` of the bulge and disk), the 
total number of likelihood evaluations will reduce. Furthermore, the simpler parameter space likely means that the 
fit will take less than 10000 likelihood evaluations per free parameter to converge. This is aided further by the reduction in `n_live` to 75.

Fits using standard light profiles and linear light profiles therefore take roughly the same time to run. However,
the simpler parameter space of linear light profiles means that the model-fit is more reliable, less susceptible to
converging to an incorrect solution and scales better if even more light profiles are included in the model.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

__Result__

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

This confirms that `intensity` parameters are not inferred by the model-fit.

In [ ]:
print(result.info)

We plot the maximum likelihood fit, galaxy images and posteriors inferred via Nautilus.

The galaxy bulge and disk appear similar to those in the data, confirming that the `intensity` values inferred by
the inversion process are accurate.

In [ ]:
print(result.max_log_likelihood_instance)

aplt.subplot_galaxies(galaxies=result.max_log_likelihood_galaxies, grid=result.grids.lp)

aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)


__Intensities__

The intensities of linear light profiles are not a part of the model parameterization and therefore are not displayed
in the `model.results` file.

To extract the `intensity` values of a specific component in the model, we use the `max_log_likelihood_galaxies`,
which has already performed the inversion and therefore the galaxy light profiles have their solved for
`intensity`'s associated with them.

In [ ]:
galaxies = result.max_log_likelihood_galaxies

print(galaxies[0].bulge.intensity)

The `Galaxies` contained in the `max_log_likelihood_fit` also has the solved for `intensity` values:

In [ ]:
fit = result.max_log_likelihood_fit

galaxies = fit.galaxies

print(galaxies[0].bulge.intensity)

__Visualization__

Linear light profiles and objects containing them (e.g. galaxies) cannot be plotted because they do not 
have an `intensity` value.

Therefore, the objects created above which replaces all linear light profiles with ordinary light profiles must be
used for visualization:

In [ ]:
galaxies = result.max_log_likelihood_galaxies

aplt.plot_array(array=galaxies.image_2d_from(grid=dataset.grid), title="Image")

aplt.plot_array(array=galaxies[0].image_2d_from(grid=dataset.grid), title="Image")

__Wrap Up__

Checkout `autogalaxy_workspace/*/guides/results` for a full description of analysing results.

__Result (Advanced)__

The code belows shows all additional results that can be computed from a `Result` object following a fit with a
linear light profile.

__Max Likelihood Inversion__

As seen elsewhere in the workspace, the result contains a `max_log_likelihood_fit`, which contains the
`Inversion` object we need.

In [ ]:
inversion = result.max_log_likelihood_fit.inversion

This `Inversion` can be used to plot the reconstructed image of specifically all linear light profiles.

In [ ]:
pass  # inversion plotter replaced by standalone functions when needed

__Linear Objects (Internal Source Code)__

An `Inversion` contains all of the linear objects used to reconstruct the data in its `linear_obj_list`.

This list may include the following objects:

 - `LightProfileLinearObjFuncList`: Holds a list of linear light profiles and the functionality used to
   reconstruct data in an inversion. It may contain a single light profile (e.g. `lp_linear.Sersic`) or
   many light profiles combined in a `Basis` (e.g. `lp_basis.Basis`).

 - `Mapper`: The linear object used by a `Pixelization` to reconstruct data via an `Inversion`. The `Mapper`
   is specific to the `Pixelization`'s `Mesh` (e.g. a `RectangularMapper` is used for a `RectangularBilinearAdaptDensity`
   mesh).

In this example, the model uses one linear `Sersic` for the galaxy's bulge and one linear `Exponential` for
the galaxy's disk. The inversion therefore has two `LightProfileLinearObjFuncList` entries, one for each
linear light profile.

In [ ]:
print(inversion.linear_obj_list)

To extract results from an inversion many quantities come in lists or require us to specify the linear object
we wish to use. Knowing what linear objects are in the `linear_obj_list`, and what indexes they correspond to,
is therefore important.

In [ ]:
print(
    f"LightProfileLinearObjFuncList (Bulge Sersic)     = {inversion.linear_obj_list[0]}"
)
print(
    f"LightProfileLinearObjFuncList (Disk Exponential) = {inversion.linear_obj_list[1]}"
)

Each `LightProfileLinearObjFuncList` contains a `light_profile_list`. For both entries in this example the
list has a single light profile.

In [ ]:
print(
    f"Linear Light Profile list (Bulge Sersic)     = {inversion.linear_obj_list[0].light_profile_list}"
)
print(
    f"Linear Light Profile list (Disk Exponential) = {inversion.linear_obj_list[1].light_profile_list}"
)

__Intensities (Internal Source Code)__

The intensities of linear light profiles are not a part of the model parameterization and therefore cannot be
accessed in the resulting galaxies, as seen in previous tutorials, for example:

galaxies = result.max_log_likelihood_galaxies
intensity = galaxies[0].bulge.intensity

The intensities are also only computed once a fit is performed, as they must first be solved for via linear algebra. 
They are therefore accessible via the `Fit` and `Inversion` objects, for example as a dictionary mapping every
linear light profile (defined above) to the intensity values:

In [ ]:
fit = result.max_log_likelihood_fit

print(fit.linear_light_profile_intensity_dict)

To extract the `intensity` values of a specific component in the model, we use that component as defined in the
`max_log_likelihood_galaxies`.

In [ ]:
galaxies = fit.galaxies

bulge = galaxies[0].bulge
disk = galaxies[0].disk

print(fit.linear_light_profile_intensity_dict)

print(
    f"\n Intensity of bulge (lp_linear.Sersic) = {fit.linear_light_profile_intensity_dict[bulge]}"
)
print(
    f"\n Intensity of disk (lp_linear.Exponential) = {fit.linear_light_profile_intensity_dict[disk]}"
)

A `Plane` where all linear light profile objects are replaced with ordinary light profiles using the solved 
for `intensity` values is also accessible.

For example, the linear light profile `Sersic` of the `bulge` component above has a solved for `intensity` of ~0.75. 

The `Plane` created below instead has an ordinary light profile with an `intensity` of ~0.75.

In [ ]:
galaxies = fit.model_obj_linear_light_profiles_to_light_profiles

print(
    f"Intensity via Plane With Ordinary Light Profiles = {galaxies[0].bulge.intensity}"
)
